# Task 5 — Executed Spark and MongoDB evidence

This notebook checks the committed MongoDB aggregation and Spark checkpoint/restart summary. It reads no live service during the documentation build.

In [1]:
from pathlib import Path
import json
from IPython.display import display

def find_evidence():
    start = Path.cwd().resolve()
    for root in (start, *start.parents):
        candidate = root / 'docs' / 'evidence' / 'live_pipeline_summary.json'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('docs/evidence/live_pipeline_summary.json not found')

evidence_path = find_evidence()
evidence = json.loads(evidence_path.read_text(encoding='utf-8'))
task = evidence['task5']
display({
    'captured_at': evidence['captured_at'],
    'mongodb': task['mongodb'],
    'checkpoint': task['checkpoint'],
    'restart': task['restart'],
})

{'captured_at': '2026-07-25T03:30:43.726347Z',
 'mongodb': {'lerobot_documents': 490,
  'distinct_lerobot_file_ids': 490,
  'duplicate_file_id_groups': 0,
  'collection_documents': 493,
  'target_documents': 1,
  'target_id_equals_file_id': True,
  'target_hash': '6c0a72b26999ff1fbaa6ba5a6a074f86e7b1e9490320093885335ebaae92f4b7',
  'target_nodes': 81,
  'target_edges': {'ast': 80, 'cfg': 6, 'dfg': 1, 'call': 1}},
 'checkpoint': {'location': 'checkpoints/person3-final-docker',
  'latest_committed_batch': 124,
  'committed_offsets': {'0': 178, '1': 175, '2': 147},
  'committed_offset_total': 500,
  'pending_batches': []},
 'restart': {'resumed_at_batch': 125,
  'committed_offsets_equal_available': True,
  'stream_idle_without_new_data': True,
  'target_document_unchanged': True,
  'collection_fingerprint_unchanged': True,
  'checkpoint_unchanged': True}}

In [2]:
mongo = task['mongodb']
checkpoint = task['checkpoint']
restart = task['restart']
assert mongo['lerobot_documents'] == mongo['distinct_lerobot_file_ids'] == 490
assert mongo['duplicate_file_id_groups'] == 0
assert mongo['collection_documents'] == 490 + evidence['provenance']['fixture_documents_retained']
assert mongo['target_documents'] == 1
assert mongo['target_id_equals_file_id'] is True
assert sum(mongo['target_edges'].values()) == 88
assert checkpoint['latest_committed_batch'] == 124
assert sum(checkpoint['committed_offsets'].values()) == checkpoint['committed_offset_total'] == 500
assert checkpoint['pending_batches'] == []
assert restart['resumed_at_batch'] == checkpoint['latest_committed_batch'] + 1
assert all(restart[key] is True for key in (
    'committed_offsets_equal_available',
    'stream_idle_without_new_data',
    'target_document_unchanged',
    'collection_fingerprint_unchanged',
    'checkpoint_unchanged',
))
display({
    'status': 'PASS',
    'lerobot_documents_equals_distinct_ids': '490 = 490',
    'duplicate_groups': 0,
    'restart_batch': 125,
    'committed_equals_available_offsets': checkpoint['committed_offsets'],
})

{'status': 'PASS',
 'lerobot_documents_equals_distinct_ids': '490 = 490',
 'duplicate_groups': 0,
 'restart_batch': 125,
 'committed_equals_available_offsets': {'0': 178, '1': 175, '2': 147}}

## Reflection

The checkpoint skips normally committed offsets, while replace/upsert by stable `_id` protects the database if a micro-batch is retried after a partial failure. Reusing the same checkpoint location is part of the acceptance condition.